# ADSP 31013 IP01 Big Data and Cloud Computing

## Group Project – NYC Yellow Taxi Trips (2024/01/01 - 2025/10/15)

### Joining Yellow Taxi Trips with Hourly Meteostat Weather

This notebook is part of the group 8 project for ADSP 31013 IP01 Big Data and Cloud Computing. Its goal is to construct an enriched NYC yellow taxi dataset that combines trip-level information with hourly weather observations at pickup and dropoff locations.

We start from two inputs:

1. **NYC TLC Yellow Taxi Trip Records**:
* **VendorID** — A code indicating the TPEP provider that provided the record.
1 = Creative Mobile Technologies, LLC
2 = Curb Mobility, LLC
6 = Myle Technologies Inc
7 = Helix
* **tpep_pickup_datetime** — The date and time when the meter was engaged.
* **tpep_dropoff_datetime** — The date and time when the meter was disengaged.
* **passenger_count** — The number of passengers in the vehicle.
* **trip_distance** — The elapsed trip distance in miles reported by the taximeter.
* **RatecodeID** — The final rate code in effect at the end of the trip.
1 = Standard rate
2 = JFK
3 = Newark
4 = Nassau or Westchester
5 = Negotiated fare
6 = Group ride
99 = Null/unknown
* **store_and_fwd_flag** — Indicates whether the trip record was held in vehicle memory before sending to the vendor (“store and forward”), usually due to lack of server connection.
Y = store and forward trip
N = not a store and forward trip
* **PULocationID** — TLC Taxi Zone in which the taximeter was engaged (pickup location).
* **DOLocationID** — TLC Taxi Zone in which the taximeter was disengaged (dropoff location).
* **payment_type** — A numeric code signifying how the passenger paid for the trip.
0 = Flex Fare trip
1 = Credit card
2 = Cash
3 = No charge
4 = Dispute
5 = Unknown
6 = Voided trip

* **fare_amount** — The time-and-distance fare calculated by the meter.
* **extra** — Miscellaneous extras and surcharges.
* **mta_tax** — Tax that is automatically triggered based on the metered rate in use.
* **tip_amount** — Tip amount. This field is automatically populated for credit card tips; cash tips are **not** included.
* **tolls_amount** — Total amount of all tolls paid during the trip.
* **improvement_surcharge** — Improvement surcharge assessed at the flag drop. This surcharge has been in effect since 2015.
* **total_amount** — Total amount charged to passengers (excluding cash tips).
* **congestion_surcharge** — Total amount collected in the trip for NYS congestion surcharge.
* **airport_fee** — Fee applied for pickups at LaGuardia and John F. Kennedy Airports only.
* **cbd_congestion_fee** — Per-trip charge for MTA’s Congestion Relief Zone (effective January 5, 2025).

2. **Hourly Weather Data from Meteostat** (retrieved via the Meteostat hourly point API and preprocessed into a table with one row per hour per borough-zone):
* **time** — The timestamp of the weather observation (UTC or local-converted depending on the station’s reporting format).
* **LocationID** — The mapped NYC Taxi Zone ID associated with the weather station.
* **Borough** — The corresponding borough for the weather station.
* **Zone** — The NYC TLC taxi zone name matching the station region.
* **temp** — Air temperature (°C).
* **dwpt** — Dew point temperature (°C).
* **rhum** — Relative humidity (%).
* **prcp** — Precipitation amount (mm).
* **snow** — Snowfall amount.
* **wdir** — Wind direction (degrees).
* **wspd** — Wind speed (m/s).
* **wpgt** — Peak wind gust (m/s).
* **pres** — Atmospheric pressure (hPa).
* **tsun** — Sunshine duration.
* **coco** — Weather condition code indicating cloudiness or other meteorological phenomena.

In this notebook, we:

1. Load the raw taxi and weather datasets into PySpark.
2. Create rounded hourly timestamps for pickup and dropoff times.
3. Join taxi trips to the corresponding pickup and dropoff weather based on TLC zone and hour.
4. Produce a final **taxi + weather** dataset that will be used in downstream EDA and machine-learning notebooks (fare prediction and tip percentage classification).

**Author(s):** Zihao Huang(zihaoh3@uchicago.edu), Ray Gong(ruigong1307@uchicago.edu), Leo Liu(jliu888@uchicago.edu), Wade Chen(wangdi@uchicago.edu), Tom Chen(cychen1@uchicago.edu)

### 0. Imports and Spark Session

This part imports PySpark utilities and helper libraries, and initializes a `SparkSession` configured for our project. All subsequent transformations on taxi and weather data are executed through this session.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [2]:
from IPython.core.display import HTML
display(HTML("<style>pre { white-space: pre !important; }</style>"))

### 1. Define Paths and Load Taxi & Weather Data

This part defines the cloud storage locations (e.g., GCS bucket paths) for the raw yellow taxi trips and the Meteostat weather data, then reads them into DataFrames `taxi_df` and `weather_df` using PySpark.

In [3]:
spark = SparkSession.builder.appName("join_taxi_weather").getOrCreate()

In [4]:
bucket_url = "gs://msca-bdp-student-gcs/group_8_project/datasets/"

taxi_df = (
    spark.read.option("header", "true").option("inferSchema", "true")
    .csv(bucket_url + "raw/yellow_taxi/csv/")
)

zone_df = (
    spark.read.option("header", "true").option("inferSchema", "true")
    .csv(bucket_url + "taxi_zone.csv")
)

weather_df = (
    spark.read.option("header", "true").option("inferSchema", "true")
    .csv(bucket_url + "raw/weather/")
)

### 2. Quick Row Counts and Sanity Check

Here we compute `.count()` on `taxi_df` and `weather_df` to confirm that both datasets were loaded correctly and to get a rough sense of their scale before doing any joins.

In [5]:
# num_parts = 200

# taxi_df = taxi_df.repartition(num_parts, "PULocationID").cache()
# weather_df = weather_df.repartition(num_parts, "LocationID").cache()

taxi_df.count()
weather_df.count()

4172878

### 3. Inspect Sample Taxi Records

This part displays a small number of taxi rows, including pickup/dropoff timestamps, locations, fares, and tip amounts. This helps verify that column names, data types, and basic values look consistent with the TLC data dictionary.

In [6]:
taxi_df.show(5, truncate=False)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|7       |2025-05-01 00:01:22 |2025-05-01 00:01:22  |1              |5.01         |1         |N                 |148         |142         |1           |19.8       |0.0  |0.5    |6.39     

### 4. Inspect Sample Weather Records

This part prints a few rows from `weather_df` to confirm the structure of the weather table: hourly `time`, `LocationID`, `Borough`, `Zone`, and the Meteostat features such as `temp`, `dwpt`, `rhum`, `prcp`, `snow`, `wdir`, `wspd`, `wpgt`, `pres`, `tsun`, and `coco`.

In [7]:
weather_df.show(5, truncate=False)

+-------------------+----------+---------+---------------+----+----+----+----+----+-----+----+----+------+----+----+
|time               |LocationID|Borough  |Zone           |temp|dwpt|rhum|prcp|snow|wdir |wspd|wpgt|pres  |tsun|coco|
+-------------------+----------+---------+---------------+----+----+----+----+----+-----+----+----+------+----+----+
|2024-01-01 00:00:00|148       |Manhattan|Lower East Side|6.0 |-1.9|57.0|0.0 |null|260.0|11.0|null|1017.0|null|3.0 |
|2024-01-01 01:00:00|148       |Manhattan|Lower East Side|5.6 |-2.2|57.0|0.0 |null|260.0|11.2|null|1016.4|null|3.0 |
|2024-01-01 02:00:00|148       |Manhattan|Lower East Side|5.6 |-1.8|59.0|0.0 |null|260.0|9.4 |null|1016.4|null|3.0 |
|2024-01-01 03:00:00|148       |Manhattan|Lower East Side|5.6 |-1.1|62.0|0.0 |null|250.0|9.4 |null|1016.4|null|3.0 |
|2024-01-01 04:00:00|148       |Manhattan|Lower East Side|5.6 |-0.7|64.0|0.0 |null|260.0|9.4 |null|1016.5|null|3.0 |
+-------------------+----------+---------+---------------+----+-

### 5. Create Rounded Pickup and Dropoff Hour Keys

In this part we derive hourly keys (e.g., `pu_time` and `do_time`) by rounding the original pickup and dropoff timestamps to the nearest hour. These rounded timestamps will be used to align trips with the corresponding hourly weather observations.

In [8]:
spark.conf.set("spark.sql.session.timeZone", "America/New_York")

taxi_df = (
    taxi_df
    .withColumn("pu_time", F.date_trunc("hour", F.col("tpep_pickup_datetime") + F.expr("INTERVAL 30 MINUTES")))
    .withColumn("do_time", F.date_trunc("hour", F.col("tpep_dropoff_datetime") + F.expr("INTERVAL 30 MINUTES")))
)

In [9]:
taxi_df.show(5, truncate=False)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-------------------+-------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|pu_time            |do_time            |
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-------------------+-------------------+
|7       |2025-04-30 20:01:22 |2025-04-30 20:01:22  |1             

### 6. Prepare Pickup Weather Lookup Table (`pu_weather_df`)

This part constructs a pickup-side weather lookup table by:

* Selecting relevant weather columns,
* Renaming `time` to `pu_time` and `LocationID` to `PULocationID`,
* Prefixing weather variables with `pu_` (e.g., `temp → pu_temp`, `rhum → pu_rhum`, etc.),
* Keeping `pu_borough` and `pu_zone` as readable geographic labels.

We then preview `pu_weather_df` to ensure the schema and example rows are correct.

In [10]:
pu_weather_df = (
    weather_df
    .select(
        F.col("LocationID").alias("PULocationID"),
        F.col("time").alias("pu_time"),
        F.col("Borough").alias("pu_borough"),
        F.col("Zone").alias("pu_zone"),
        F.col("temp").alias("pu_temp"),
        F.col("dwpt").alias("pu_dwpt"),
        F.col("rhum").alias("pu_rhum"),
        F.col("prcp").alias("pu_prcp"),
        F.col("snow").alias("pu_snow"),
        F.col("wdir").alias("pu_wdir"),
        F.col("wspd").alias("pu_wspd"),
        F.col("wpgt").alias("pu_wpgt"),
        F.col("pres").alias("pu_pres"),
        F.col("tsun").alias("pu_tsun"),
        F.col("coco").alias("pu_coco"),
    )
)

In [11]:
pu_weather_df.show(5, truncate=False)

+------------+-------------------+----------+---------------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+
|PULocationID|pu_time            |pu_borough|pu_zone        |pu_temp|pu_dwpt|pu_rhum|pu_prcp|pu_snow|pu_wdir|pu_wspd|pu_wpgt|pu_pres|pu_tsun|pu_coco|
+------------+-------------------+----------+---------------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+
|148         |2024-01-01 00:00:00|Manhattan |Lower East Side|6.0    |-1.9   |57.0   |0.0    |null   |260.0  |11.0   |null   |1017.0 |null   |3.0    |
|148         |2024-01-01 01:00:00|Manhattan |Lower East Side|5.6    |-2.2   |57.0   |0.0    |null   |260.0  |11.2   |null   |1016.4 |null   |3.0    |
|148         |2024-01-01 02:00:00|Manhattan |Lower East Side|5.6    |-1.8   |59.0   |0.0    |null   |260.0  |9.4    |null   |1016.4 |null   |3.0    |
|148         |2024-01-01 03:00:00|Manhattan |Lower East Side|5.6    |-1.1   |62.0   |0.0    |null   

### 7. Join Taxi Trips with Pickup Weather

Here we join `taxi_df` with `pu_weather_df` on `PULocationID` and the rounded pickup time (`pu_time`). This enriches each trip with the weather conditions at the time and TLC zone where the passenger was picked up.

In [12]:
pu_taxi_weather_df = (
    taxi_df
    .join(pu_weather_df, on=["PULocationID", "pu_time"], how="left")
)

In [13]:
pu_taxi_weather_df.where(F.col("PULocationID") == 148).show(5, truncate=False)

25/11/18 07:13:19 WARN org.apache.spark.sql.catalyst.util.package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+------------+-------------------+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-------------------+----------+---------------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+
|PULocationID|pu_time            |VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|do_time            |pu_borough|pu_zone        |pu_temp|pu_dwpt|pu_rhum|pu_prcp|pu_snow|pu_wdir|pu_wspd|pu_wpgt|pu_pres|pu_tsun|pu_coco|
+------------+-------------------+--------+--------------------+---------------------+---------------+-------------+----------+------------------+--

### 8. Prepare Dropoff Weather Lookup Table (`do_weather_df`)

This cell mirrors the pickup logic for dropoffs:

* Renaming `time` to `do_time` and `LocationID` to `DOLocationID`,
* Prefixing weather variables with `do_` (e.g., `temp → do_temp`, `rhum → do_rhum`, etc.),
* Keeping `do_borough` and `do_zone` for the dropoff area.

We show a few rows to verify the structure of `do_weather_df`.

In [14]:
do_weather_df = (
    weather_df
    .select(
        F.col("LocationID").alias("DOLocationID"),
        F.col("Borough").alias("do_borough"),
        F.col("Zone").alias("do_zone"),
        F.col("time").alias("do_time"),
        F.col("temp").alias("do_temp"),
        F.col("dwpt").alias("do_dwpt"),
        F.col("rhum").alias("do_rhum"),
        F.col("prcp").alias("do_prcp"),
        F.col("snow").alias("do_snow"),
        F.col("wdir").alias("do_wdir"),
        F.col("wspd").alias("do_wspd"),
        F.col("wpgt").alias("do_wpgt"),
        F.col("pres").alias("do_pres"),
        F.col("tsun").alias("do_tsun"),
        F.col("coco").alias("do_coco"),
    )
)

### 9. Join Taxi + Pickup Weather with Dropoff Weather

In this part we take the intermediate **taxi + pickup weather** table and join it with `do_weather_df` using `DOLocationID` and the rounded dropoff time (`do_time`). The result is a fully enriched dataset that carries both pickup-side and dropoff-side weather features for every trip.

In [15]:
taxi_with_both_weather = (
    pu_taxi_weather_df
    .join(do_weather_df, on=["DOLocationID", "do_time"], how="left")
)

In [16]:
taxi_with_both_weather.where(F.col("DOLocationID") == 148).show(5, truncate=False)

+------------+-------------------+------------+-------------------+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+----------+-----------------------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+----------+---------------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+
|DOLocationID|do_time            |PULocationID|pu_time            |VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|pu_borough|pu_zone                |pu_temp|pu_dwpt|pu_rhum|pu_prcp|pu_snow|pu_wdir|pu_wspd|pu_wpgt|pu_pres|pu_tsun|pu_coco|do_borough|do_zone

### 10. Inspect Final Joined Schema and Sample Rows

This part prints the schema (or a sample of rows) of the final joined DataFrame to confirm that:

* All original taxi columns (timestamps, locations, fare, tip, etc.) are preserved,
* Pickup weather columns are correctly prefixed with `pu_`,
* Dropoff weather columns are correctly prefixed with `do_`,
* Borough/zone labels are attached for both PULocationID and DOLocationID.
* Re-arrange the columns order and rename them so that they are aligned

In [17]:
taxi_df = taxi_with_both_weather.select(
    ["tpep_pickup_datetime", "tpep_dropoff_datetime",
     F.col("PULocationID").alias("pu_location_id"), F.col("DOLocationID").alias("do_location_id"),
     "pu_borough", "pu_zone", "do_borough", "do_zone",
     F.col("VendorID").alias("vendor_id"), F.col("RatecodeID").alias("ratecode_id"),
     "passenger_count", "trip_distance", "store_and_fwd_flag",
     "payment_type", "fare_amount", "extra", "mta_tax", "tip_amount",
     "tolls_amount", "improvement_surcharge", "congestion_surcharge", F.col("Airport_fee").alias("airport_fee"), "total_amount",
     "pu_temp", "pu_dwpt", "pu_rhum", "pu_prcp", "pu_snow", "pu_wdir", "pu_wspd", "pu_wpgt", "pu_pres", "pu_tsun", "pu_coco",
     "do_temp", "do_dwpt", "do_rhum", "do_prcp", "do_snow", "do_wdir", "do_wspd", "do_wpgt", "do_pres", "do_tsun", "do_coco"]
)

In [18]:
taxi_df.show(5, truncate=False)

+--------------------+---------------------+--------------+--------------+----------+--------------------+----------+-------+---------+-----------+---------------+-------------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+--------------------+-----------+------------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+-------+
|tpep_pickup_datetime|tpep_dropoff_datetime|pu_location_id|do_location_id|pu_borough|pu_zone             |do_borough|do_zone|vendor_id|ratecode_id|passenger_count|trip_distance|store_and_fwd_flag|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|congestion_surcharge|airport_fee|total_amount|pu_temp|pu_dwpt|pu_rhum|pu_prcp|pu_snow|pu_wdir|pu_wspd|pu_wpgt|pu_pres|pu_tsun|pu_coco|do_temp|do_dwpt|do_rhum|do_prcp|do_snow|do_wdir|do_wspd|do_wpgt|do_pres|do_tsun|d

### 11. Write Joined Taxi–Weather Dataset to Cloud Storage

The final cell writes the enriched **taxi + weather** table back to cloud storage (e.g., as csv/Parquet files). This output serves as the “feature-ready” dataset that will be consumed by subsequent EDA and machine-learning notebooks for fare prediction and tip percentage classification.

In [19]:
(
    taxi_df
    .write
    .mode("overwrite")
    .option("header", "true")
    .option("quote", '"')
    .option("maxRecordsPerFile", 5_000_000)
    .csv(bucket_url + "raw/yellow_taxi_weather_joined/csv")
)

In [20]:
(
    taxi_df
    .write
    .mode("overwrite")
    .option("maxRecordsPerFile", 5_000_000)
    .parquet(bucket_url + "raw/yellow_taxi_weather_joined/parquet")
)

## Conclusion

By the end of this notebook, we obtain a joined dataset where each yellow taxi trip is augmented with hourly Meteostat weather at both pickup and dropoff.

* The **core taxi trip fields** include timestamps (`tpep_pickup_datetime`, `tpep_dropoff_datetime`), pickup and dropoff zones (`pu_location_id`, `do_location_id`, `pu_borough`, `pu_zone`, `do_borough`, `do_zone`), trip characteristics (`passenger count`, `distance`, `rate code`, `payment type`), and monetary outcomes (`fare`, `surcharges`, `tips`, `tolls`, `total amount`).
* The **pickup weather features** are stored as `pu_*` columns such as `pu_time`, `pu_temp`, `pu_dwpt`, `pu_rhum`, `pu_prcp`, `pu_snow`, `pu_wdir`, `pu_wspd`, `pu_wpgt`, `pu_pres`, `pu_tsun`, and `pu_coco`.
* The **dropoff weather features** mirror this structure with `do_*` prefixes: `do_time`, `do_temp`, `do_dwpt`, `do_rhum`, `do_prcp`, `do_snow`, `do_wdir`, `do_wspd`, `do_wpgt`, `do_pres`, `do_tsun`, and `do_coco`.